# Trust Analysis of Traffic Sign Classifiers - Training

This notebook facilitates running the training pipeline for the TSR Thesis project on Google Colab with GPU acceleration.

## 1. Setup Environment

In [ ]:
# Clone the repository
!git clone https://github.com/ucefhesham/tsr-thesis.git
%cd tsr-thesis

# Install dependencies
!pip install -r requirements.txt

# Install additional colab-specific tools
!pip install wandb -U

## 2. Weights & Biases Logging
Logging into W&B is required to track experiments and sync model checkpoints back to your local machine.

In [ ]:
import wandb
wandb.login()

## 3. Run Training
Execute the training script. Results will be saved to W&B and can be downloaded locally from the W&B Artifacts dashboard.

In [ ]:
# Train with default configuration (ResNet18 on GTSRB)
!python train.py trainer.max_epochs=20 trainer.devices=1 trainer.accelerator='gpu'

## 4. Retrieve Results
Checkpoints are automatically uploaded to W&B. 

To download a checkpoint back to your local machine, use the following code in your local terminal:

```bash
wandb artifact get ucefhesham-universit-t-ulm/trust-analysis-tsr/model-<run_id>:v0 --root ./outputs/checkpoints
```

## 4.5 Download Trained Model
Since large model weights are not stored in GitHub, we download them directly from W&B Artifacts.

In [ ]:
# --- DOWNLOAD YOUR MODEL FROM W&B ---
# Replace <run_id> with your actual W&B run ID (e.g., qkfx from your last logs)
# This retrieves the 'best' model checkpoint you trained earlier.

import os
os.makedirs('checkpoints', exist_ok=True)

!wandb artifact get ucefhesham-universit-t-ulm/trust-analysis-tsr/model-qkfx:v0 --root ./checkpoints

## 5. Phase 4: Uncertainty Stress Suite

Evaluate the model's robustness against digital and environmental corruptions (Noise, Blur, Weather, Compression) across 5 severity levels. This automated sweep exports results to CSV for thesis analysis.

In [ ]:
# Run the automated Stress Suite sweep
!python eval.py ckpt_path="checkpoints/best-trust-baseline.ckpt" trainer.devices=1 trainer.accelerator='gpu'

# Download the results CSV to your local machine
from google.colab import files
files.download('logs/stress_test_results.csv')

## 6. Phase 5: Post-Hoc Calibration

Optimize model reliability using Temperature Scaling. This step scales the logits to minimize Cross-Entropy Loss on a validation set, typically improving ECE (Expected Calibration Error).

In [ ]:
# The calibration module is integrated in src/models/calibration.py
# Example usage for future scripts:
# from src.models.calibration import ModelWithTemperature
# calibrated_model = ModelWithTemperature(model)
# calibrated_model.set_temperature(datamodule.val_dataloader())
print("Calibration infrastructure ready.")